примерное содержание проекта:

*   машина, которая определяет инфекционное заболевание по симптоматике (классификатор, машинное обучение + предобработка данных это пандас)
- мне не нужна машина мне нужна просто таблица и косинусная близость по симптомам
*   парсер который анализирует словарь с симптомами на предмет совпадений с текстом или что-то в этом духе (NLP-инструмент для русского языка)

0.   исправление опечаток +
1.   лемматизатор (симптомы) и удаление стоп-слов чтобы распарсить текст+
3.   еще можно по приколу добавить парсинг языка (чтобы он умел говорить "я умею работать только с русским языком)


*   бот который может выдать какую-то информацию о заболевании на основании запроса (БД и запросы sql); нужно еще чтобы он мог что-то неточное разобрать (типа хдлера вместо холера) *действительно хд*







In [135]:
complaint = input()

Пациент испытывает боль в мышцах и суставах, есть ослабление мышц; в анамнез имели место половые внебрачные крнтакты, но сифилис исключен по аналрзам


Установка пакетов

In [ ]:
!pip install deep_translator
!pip install autocorrect -q
!pip install -U spacy --q
!python -m spacy download ru_core_news_sm --q
!pip install nltk
!pip install pymorphy3 --q
!pip install gensim sentence-transformers

Импорты (все нужные)

In [ ]:
#обработка и перевод таблицы
import pandas as pd
from deep_translator import GoogleTranslator

#обработка вводимого текста
from autocorrect import Speller
import spacy
import nltk

from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')

from nltk.corpus import stopwords
nltk.download('stopwords')

nlp_ru = spacy.load("ru_core_news_sm")

from sklearn.metrics.pairwise import cosine_similarity
#from sklearn.feature_extraction.text import CountVectorizer

###Работа с таблицей

In [111]:
df_symptoms = pd.read_csv('DiseaseAndSymptoms.csv')

In [112]:
df_symptoms.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Шаг 0: переводим таблицу, потому что она будет работать с русским языком

In [7]:
translator = GoogleTranslator(source='en', target='ru')

In [116]:
unique_values = pd.Series(df_symptoms.values.ravel()).dropna().unique().tolist()
unique_values_2 = {}

for value in unique_values:
    new_value = value.replace('_', ' ').lstrip()
    unique_values_2[value] = new_value

df_symptoms = df_symptoms.replace(unique_values_2)

value_set = set(unique_values_2.values())

def translate_text(text):
    if pd.notna(text) and isinstance(text, str):
        return translator.translate(text)

trans_dict = {}

for value in value_set:
    trans_dict[value] = translate_text(value)

{'swelling of stomach': 'вздутие живота', 'Migraine': 'Мигрень', 'constipation': 'запор', 'dehydration': 'обезвоживание', 'Arthritis': 'Артрит', 'Fungal infection': 'Грибковая инфекция', 'Dimorphic hemmorhoids(piles)': 'Диморфные геморроидальные узлы (сваи)', 'scurring': 'несущийся', 'AIDS': 'СПИД', 'red spots over body': 'красные пятна по телу', 'lethargy': 'вялость', 'runny nose': 'насморк', 'Paralysis (brain hemorrhage)': 'Паралич (кровоизлияние в мозг)', 'blurred and distorted vision': 'затуманенное и искаженное зрение', 'silver like dusting': 'серебро, как пыль', 'fatigue': 'усталость', 'back pain': 'боль в спине', 'skin rash': 'кожная сыпь', 'pain behind the eyes': 'боль за глазами', 'depression': 'депрессия', 'muscle pain': 'мышечная боль', 'Diabetes ': 'Диабет', 'Osteoarthristis': 'Остеоартрис', 'puffy face and eyes': 'опухшее лицо и глаза', 'skin peeling': 'шелушение кожи', 'muscle weakness': 'мышечная слабость', 'pus filled pimples': 'прыщи, наполненные гноем', 'receiving blo

In [117]:
df_symptoms = df_symptoms.replace(trans_dict)

Шаг 1: сделать из этого табличку со списком симптомов

In [158]:
df_symptoms['symptoms_list'] = df_symptoms.drop('Disease', axis=1).values.tolist()
df_symptoms['symptoms_list'] = df_symptoms['symptoms_list'].apply(lambda x: [i for i in x if pd.notna(i)])
df_symptoms = df_symptoms[['Disease', 'symptoms_list']]
df_symptoms['symptoms_list'] = df_symptoms['symptoms_list'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)
df_symptoms

/tmp/ipykernel_1492/3206944067.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_symptoms['symptoms_list'] = df_symptoms.drop('Disease', axis=1).values.tolist()
/tmp/ipykernel_1492/3206944067.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_symptoms['symptoms_list'] = df_symptoms['symptoms_list'].apply(lambda x: [i for i in x if pd.notna(i)])


,Disease,symptoms_list
0,Грибковая инфекция,зуд кожная сыпь узловые высыпания на коже дисх...
1,Грибковая инфекция,кожная сыпь узловые высыпания на коже дисхромн...
2,Грибковая инфекция,зуд узловые высыпания на коже дисхромные пятна
3,Грибковая инфекция,зуд кожная сыпь дисхромные пятна
4,Грибковая инфекция,зуд кожная сыпь узловые высыпания на коже
...,...,...
4915,(головокружение) Пароймсальное позиционное гол...,рвота головная боль тошнота вращательные движе...
4916,Прыщи,"кожная сыпь прыщи, наполненные гноем угри несу..."
4917,Инфекция мочевыводящих путей,жгучее мочеиспускание дискомфорт в мочевом пуз...
4918,Псориаз,кожная сыпь боль в суставах шелушение кожи сер...


In [159]:
all_symptoms = list(set([symptoms for symptoms
                         in df_symptoms['symptoms_list']]))

###Парсинг текста

Идея бота состоит в том, что он принимает на вход цельный врачебный анализ жалобы пациента, а не дает ему шаблон или выбор из нескольких кнопок: это позволяет (*позволяло бы, если бы речь шла о настоящем медицинском проекте*) не упустить ничего, что показалось значимым врачу и о чем мог не подумать разработчик. А мне позволяет использовать NLP-инструменты:)

####Исправление опечаток

Текст вводится живым человеком, поэтому опечатки, очевидно, могут быть – вряд ли они будут в ключевых словах, но всякий случай бот умеет с этим справляться; если, например, врач выберет кнопку "Я хочу узнать о лечении и профилактике диагноза" и случайно напишет туда слово "Хдлера", бот не поймет, что тот пытался сказать, и скажет, что такой болезни у него нет – а с дополнительной проверкой он сразу выдаст результат.

In [140]:
spell = Speller(lang='ru')

dictionary for this language not found, downloading...
__________________________________________________
couldn't download http://ipfs.io/ipfs/QmbRSZvfJV6zN12zzWhecphcvE9ZBeQdAJGQ9c9ttJXzcg/ru.tar.gz, trying next url...
__________________________________________________
couldn't download https://gateway.pinata.cloud/ipfs/QmbRSZvfJV6zN12zzWhecphcvE9ZBeQdAJGQ9c9ttJXzcg/ru.tar.gz, trying next url...
__________________________________________________
couldn't download https://cf-ipfs.com/ipfs/QmbRSZvfJV6zN12zzWhecphcvE9ZBeQdAJGQ9c9ttJXzcg/ru.tar.gz, trying next url...
__________________________________________________
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
done!


In [141]:
complaint = spell(complaint)

####Подготовка текста

In [22]:
sw = stopwords.words('russian')
sw.extend(['жаловаться', 'начаться', 'появиться', 'пациент', 'симптом'])

#добавлены слова, которые идейно должны встречаться в большом количестве сообщений

In [146]:
complaint_doc = nlp_ru(complaint)
complaint_tokens = set([token.lemma_ for token in complaint_doc])
words = [w.lower() for w in word_tokenize(' '.join(complaint_tokens),
                                          language='russian') if w.isalpha()]
#issues конечно есть с этим isalpha, он не распарсит "поднялась 37.8"
complaint_filtered = [w for w in words if w not in sw]

In [27]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [149]:
print(complaint_filtered)

['анализ', 'половой', 'контакт', 'иметь', 'исключить', 'боль', 'мышца', 'испытывать', 'внебрачный', 'сустав', 'место', 'ослабление', 'сифилис', 'анамнез']


In [29]:
!pip install gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 14.6 MB/s eta 0:00:00


In [154]:
texts = [' '.join(complaint_filtered)] + all_symptoms

In [129]:
print(texts[:5])

['сустав казаться температура лихорадка серьёзный боль', 'атрофия мышц пятна в горле внебрачные контакты', 'головная боль боль в груди головокружение потеря равновесия', 'головная боль боль в груди головокружение потеря равновесия отсутствие концентрации', 'кислотность несварение желудка головная боль затуманенное и искаженное зрение чрезмерный голод скованность мышц шеи депрессия раздражительность нарушения зрения']


In [153]:
complaint_filtered

['анализ',
 'половой',
 'контакт',
 'иметь',
 'исключить',
 'боль',
 'мышца',
 'испытывать',
 'внебрачный',
 'сустав',
 'место',
 'ослабление',
 'сифилис',
 'анамнез']

In [33]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api
from gensim.models import KeyedVectors

!wget https://rusvectores.org/static/models/rusvectores4/RNC/ruscorpora_upos_skipgram_300_5_2018.vec.gz
model = KeyedVectors.load_word2vec_format('ruscorpora_upos_skipgram_300_5_2018.vec.gz')

--2026-03-25 18:25:41--  https://rusvectores.org/static/models/rusvectores4/RNC/ruscorpora_upos_skipgram_300_5_2018.vec.gz
Resolving rusvectores.org (rusvectores.org)... 129.240.189.200, 2001:700:112::200
Connecting to rusvectores.org (rusvectores.org)|129.240.189.200|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 199864398 (191M) [application/x-gzip]
Saving to: ‘ruscorpora_upos_skipgram_300_5_2018.vec.gz’

ruscorpora_upos_ski 100%[===================>] 190.61M  25.3MB/s    in 8.4s    

2026-03-25 18:25:50 (22.7 MB/s) - ‘ruscorpora_upos_skipgram_300_5_2018.vec.gz’ saved [199864398/199864398]



In [34]:
from pymorphy3 import MorphAnalyzer

morph = MorphAnalyzer()

In [161]:
tagged_list = [[f'{morph.parse(word)[0].normal_form}_{morph.parse(word)[0].tag.POS}'
                for word in list_.split(' ')]
               for list_ in texts]

In [156]:
def get_sentence_vector(text):
    #words = text.split()
    vectors = [model[word] for word in text if word in model]
    return np.mean(vectors, axis=0) if vectors else np.zeros(300)

vec = np.array([get_sentence_vector(text) for text in tagged_list])

target = vec[0].reshape(1, -1)
others = vec[1:]
similarities = cosine_similarity(target, others).flatten()
sorted_indices = np.argsort(similarities)[::-1]

disease_pred = texts[sorted_indices[0] + 1]
print(disease_pred)

атрофия мышц пятна в горле высокая температура внебрачные контакты


In [157]:
df_symptoms.loc[df_symptoms['symptoms_list'] == disease_pred, 'Disease'].iloc[0]

'СПИД'

###БД

Этот раздел может использоваться, если пользователь говорит "я хочу узнать больше о заболевании".

In [180]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('diseases.db')

conn.execute('DROP TABLE IF EXISTS diseases')
conn.execute('DROP TABLE IF EXISTS symptoms')

df_diseases = pd.DataFrame({
    'id': range(1, len(df_symptoms['Disease'].unique()) + 1),
    'name': df_symptoms['Disease'].unique()
})
df_diseases.to_sql('diseases', conn, if_exists='replace', index=False)

df_symptoms_new = pd.DataFrame({
    'id': range(1, len(df_symptoms) + 1),
    'symptoms_list': df_symptoms['symptoms_list']
})
df_symptoms_new.to_sql('symptoms', conn, if_exists='replace', index=False);



In [187]:
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS icd_who (
        disease_id INTEGER,
        mkb_link TEXT
    )
''')

df_diseases = pd.read_sql('SELECT * FROM diseases', conn)
df_mkb = pd.read_excel('/content/SimpleTabulation-ICD-11-MMS-ru.xlsx')

for idx, row in df_diseases.iterrows():
    disease_name = row['name']
    disease_id = row['id']

    match = df_mkb[df_mkb['Title'].str.contains(disease_name, case=False, na=False)]

    if not match.empty:
        cursor.execute('''
            INSERT OR REPLACE INTO icd_who (disease_id, disease_name, mkb_link)
            VALUES (?, ?, ?)
        ''', (disease_id, disease_name, match.iloc[0]['Parent']))
        conn.commit()

/tmp/ipykernel_1492/3646869670.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  match = df_mkb[df_mkb['Title'].str.contains(disease_name, case=False, na=False)]


In [ ]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS prevention (
        disease_id INTEGER,
        prevention_cautions TEXT
    )
''')
df_diseases = pd.read_sql('SELECT * FROM diseases', conn)

for _, row in df_diseases.iterrows():
    cursor.execute('''
        INSERT OR REPLACE INTO prevention (disease_id, disease_name, prevention_measures)
        VALUES (?, ?, ?)
    ''', (row['id'], row['name'], 'Пока нет информации'))

conn.commit()

In [ ]:
conn.close()

###Бот

этот раздел катастрофически не закончен

In [2]:
!pip install pyTelegramBotAPI

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 8.2 MB/s eta 0:00:00


In [3]:
!pip install aiogram nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 716.4/716.4 kB 9.0 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/chec

KeyboardInterrupt: 

In [ ]:
import telebot
from telebot.async_telebot import AsyncTeleBot
from telebot import types

In [ ]:
!pip install aiogram nest_asyncio

from aiogram import Bot, Dispatcher, types
from aiogram.filters import Command
import nest_asyncio

nest_asyncio.apply()

TOKEN = '8334838059:AAE1goQ36xVP_rMaoV_K2DFs_PRk2dLFsc4'
bot = Bot(token=TOKEN)
dp = Dispatcher()

@dp.message(Command('start'))
async def start(message: types.Message):
    # Новый способ создания клавиатуры
    keyboard = [
        [types.KeyboardButton(text="Начать работу")]
    ]
    markup = types.ReplyKeyboardMarkup(keyboard=keyboard, resize_keyboard=True)
    await message.answer('Нажмите "Начать работу"', reply_markup=markup)

@dp.message()
async def handle(message: types.Message):
    if message.text == "Начать работу":
        keyboard = [
            [types.KeyboardButton(text="Я согласен с условиями")]
        ]
        markup = types.ReplyKeyboardMarkup(keyboard=keyboard, resize_keyboard=True)
        await message.answer(
"""Здравствуйте! Я – бот-помощник в определении инфекционных заболеваний.
Пожалуйста, перед использованием ознакомьтесь с несколькими правилами:
- я предназначен только для врачебного пользования; если вы не врач, к сожалению, вы не можете воспользоваться мной.
- я не могу дать точный диагноз, а могу лишь подтвердить опасения или дать перечень симптомов заболевания;
окончательное решение о диагнозе остается за вами!
Нажимая кнопку "Я согласен с условиями", вы подтверждаете, что являетесь врачом с медицинским образованием и прочитали условия выше.""", reply_markup=markup)

    elif message.text == "Я согласен с условиями":
        keyboard = [
            [types.KeyboardButton(text="Я не знаю диагноз пациента и хочу определить его по жалобам")],
            [types.KeyboardButton(text="Я предполагаю, что знаю диагноз, и хочу его подтвердить")],
            [types.KeyboardButton(text="Я просто хочу узнать больше о конкретном заболевании")]
        ]
        markup = types.ReplyKeyboardMarkup(keyboard=keyboard, resize_keyboard=True)
        await message.answer("Выберите, пожалуйста, нужное действие", reply_markup=markup)

In [ ]:
await dp.start_polling(bot)